In [2]:
!pip install -U -q langchain-groq

In [3]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')

In [4]:
from langchain_groq import ChatGroq

model=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY
)

# RunnableParallel

In [7]:
from langchain_core.runnables import RunnableSequence, RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [10]:
prompt1=PromptTemplate(
    template="Generate a Tweet about {topic}"
)
prompt2=PromptTemplate(
    template="Generate a linkdin post for {topic}"
)
parser=StrOutputParser()

In [11]:
parallel_chain=RunnableParallel(
    {
        "tweet":prompt1 | model | parser ,
        "linkdin":prompt2 | model | parser
    }
)
result=parallel_chain.invoke({"topic":"ML"})

In [19]:
print("tweeter post\n",result['tweet'])
print()
print()
print()
print("linkdin post\n",result['linkdin'])

tweeter post
 🚀 Just trained a model that predicts your next favorite song 🎶—because why let humans decide? 🤖✨ #MachineLearning #AI #DataScience #FutureTech #MLMagic #TechTalk #Innovation 💡



linkdin post
 🚀 **Just wrapped up a game‑changing ML pilot at [Your Company]** 🚀  

After weeks of data wrangling, model tuning, and a few sleepless nights, we’re now able to predict customer churn with **92% accuracy**—a 15% lift over our previous rule‑based system.  

Key take‑aways that I’d love to share:

1. **Feature engineering matters** – a few engineered variables (e.g., “days since last upgrade” + “interaction frequency”) turned a mediocre model into a high‑performer.  
2. **Explainability is non‑optional** – using SHAP values not only boosted stakeholder confidence but also uncovered a hidden bias that we fixed before launch.  
3. **Iterative deployment** – a staged rollout with real‑time monitoring allowed us to catch a drift early and retrain before it impacted revenue.  

🔗 *Next ste

# Runable PassThrough

In [21]:
from langchain_core.runnables import RunnablePassthrough
prompt=PromptTemplate(
    template="Tell a joke about {topic}"
)

chain={"topic":RunnablePassthrough()} | prompt

chain.invoke("Tujro")

StringPromptValue(text='Tell a joke about Tujro')

In [24]:
prompt1=PromptTemplate(
    template="Write a joke about {topic}"
)

prompt2=PromptTemplate(
    template="Explain the joke:{text}"
)

joke_chain=prompt1 | model | parser

parallel_chain=RunnableParallel(
    {
        "joke":RunnablePassthrough(),
        "explanation":prompt2 | model | parser
    }
)
merge_chain=joke_chain | parallel_chain
result=merge_chain.invoke({"topic":"Z.I. Turjo"})

In [27]:
print("==============Joke==========\n")
print(result['joke'])
print()
print()
print("==============Explanation==========\n")
print(result['explanation'])

==============Joke==========

Why did Z.I. Turjo bring a ladder to the budget meeting?

Because he heard the agenda was “high” and he wanted to climb straight to the top of the fiscal cliff!


==============Explanation==========

**The joke hinges on a couple of double‑meanings that play off each other:**

| Phrase in the joke | Literal meaning | Political / fiscal meaning |
|--------------------|-----------------|----------------------------|
| **“high” agenda** | Something that is physically high up (e.g., a high shelf) | An agenda that is ambitious, lofty, or “high‑level” in terms of policy goals. |
| **“climb straight to the top of the fiscal cliff”** | Literally climbing a steep drop with a ladder | “Fiscal cliff” is a term for a situation where a bunch of tax cuts and spending cuts are set to take effect at the same time, creating a steep drop in government revenue or a rapid rise in deficits. The “top” of that cliff would be the point before the drop begins. |

**Putting it toge

# RunnableLambda

In [28]:
from langchain_core.runnables import RunnableLambda

upper=RunnableLambda(lambda x: x.upper())

upper.invoke("Hey this is Turjo ")

'HEY THIS IS TURJO '

In [37]:
promt1=PromptTemplate(
    template="Generate joke about {topic}"
)

joke_gen=prompt1 | model | parser

parallel_chain= RunnableParallel(
    {
        "joke":RunnablePassthrough(),
        "word_count": RunnableLambda(lambda x : len(x.strip()))
    }
)
merge_chain=joke_chain | parallel_chain

result=merge_chain.invoke({"topic":"ML"})

In [40]:
print(result['joke'])

Why did the machine learning model break up with its dataset?

Because it kept saying, “I’m just a *supervised* relationship—no *unsupervised* fun for me!”


In [39]:
result['word_count']

155

In [43]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

# 1. Define your destination components
physics_chain = ChatPromptTemplate.from_template("You are a physics expert. Answer: {query}")
math_chain = ChatPromptTemplate.from_template("You are a math expert. Answer: {query}")
general_chain = ChatPromptTemplate.from_template("Answer this general query: {query}")

# 2. Create the routing function
def route_query(inputs):
    topic = inputs["topic"].lower()
    if "physics" in topic:
        return physics_chain
    elif "math" in topic:
        return math_chain
    else:
        return general_chain

# 3. Build the final chain
# LangChain automatically converts the routing function to a RunnableLambda
full_chain = RunnableLambda(route_query)

# 4. Execute
print(full_chain.invoke({"topic": "math", "query": "What is 2+2?"}))

messages=[HumanMessage(content='You are a math expert. Answer: What is 2+2?', additional_kwargs={}, response_metadata={})]
